In [61]:
!pip install -q langchain-groq
%pip install -q langchain-community
%pip install -q langchain-classic

In [76]:
import os, requests
from google.colab import userdata
from langchain_groq import ChatGroq
from pathlib import Path
from langchain_core.prompts import (ChatPromptTemplate,
                                    HumanMessagePromptTemplate,
                                    SystemMessagePromptTemplate,
                                    AIMessagePromptTemplate,
                                    MessagesPlaceholder
                                    )
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_classic.chains.llm import LLMChain
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory
from langchain_classic.globals import set_verbose


set_verbose(True)

In [63]:
def _read_env_file(name, filename="secrets.env"):
    """Look for KEY=value in secrets.env in the working dir or /content."""
    for folder in (Path.cwd(), Path("/content")):
        path = folder / filename
        if path.is_file():
            for line in path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, val = line.partition("=")
                if key.strip().removeprefix("export ").strip() == name:
                    return val.strip().strip('"').strip("'")
    return None

def get_secret(name, required=True):
    value = None
    try:
        value = userdata.get(name)      # works in the Colab web UI
    except Exception:
        pass                            # timeout / not found / no access
    if not value:
        value = _read_env_file(name)    # fallback: uploaded secrets.env
    if value:
        os.environ[name] = value
    elif required:
        raise RuntimeError(
            f"{name} not found in Colab Secrets or secrets.env. "
            "Upload secrets.env to the Colab server and re-run."
        )
    return value

groq_key = get_secret("GROQ_API_KEY")


print("GROQ key loaded:", bool(groq_key))


GROQ key loaded: True


In [64]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    model_kwargs= {"seed" : 350 },
    temperature= 0,
    max_tokens= 250
)

chatmessage history

In [65]:
human_template = HumanMessagePromptTemplate.from_template('{question}')
system_template = SystemMessagePromptTemplate.from_template("you are a sarcastic chatbot that answers in sarcastic way")

chat_template = ChatPromptTemplate.from_messages( background_info.messages + [human_template])

print(chat_template.messages)

[HumanMessage(content='whats the biggest planet of our solar system ?', additional_kwargs={}, response_metadata={}), AIMessage(content='Jupiter. It’s more than twice as massive as all the other planets in the solar system combined', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [66]:
background_info = ChatMessageHistory()
background_info.add_user_message('whats the biggest planet of our solar system ?')
background_info.add_ai_message('Jupiter. It’s more than twice as massive as all the other planets in the solar system combined')

In [67]:
chain = LLMChain(
    llm = llm,
    prompt= chat_template
)

In [68]:
response =  chain.invoke({'question' : 'how wide it is than earth ?'})

print(response['text'])



> Entering new LLMChain chain...
Prompt after formatting:
Human: whats the biggest planet of our solar system ?
AI: Jupiter. It’s more than twice as massive as all the other planets in the solar system combined
Human: how wide it is than earth ?

> Finished chain.
**Jupiter vs. Earth – size comparison**

| Property | Jupiter | Earth | How many × bigger than Earth |
|----------|---------|-------|------------------------------|
| **Mean diameter** (equatorial) | **139 820 km** | 12 742 km | **≈ 11 times** |
| **Mean radius** | 69 911 km | 6 371 km | **≈ 11 times** |
| **Polar diameter** (because Jupiter is slightly flattened) | 133 708 km | — | **≈ 10.5 times** |
| **Volume** | 1.43 × 10¹⁵ km³ | 1.08 × 10¹² km³ | **≈ 1 300 times**


**Buffer Memory**

In [69]:
system_template = SystemMessagePromptTemplate.from_template("you are a sarcastic chatbot that answers in sarcastic way")
human_template = HumanMessagePromptTemplate.from_template(template = "{question}")
history_placeholder = MessagesPlaceholder(variable_name= "message_log")

In [70]:
chat_template = ChatPromptTemplate.from_messages(messages= [system_template,history_placeholder,human_template])

In [71]:
message_history = ChatMessageHistory()
message_history.add_user_message("whats the biggest planet of our solar system ?")
message_history.add_ai_message("Jupiter. Shocking, I know.")

In [72]:
memory = ConversationBufferMemory(
    memory_key= "message_log",
    chat_memory= message_history,
    return_messages= True
)

memory.load_memory_variables({})["message_log"]

[HumanMessage(content='whats the biggest planet of our solar system ?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Jupiter. Shocking, I know.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [73]:
chain = LLMChain(
    llm = llm,
    memory = memory,
    prompt= chat_template
)

response = chain.invoke({"question": "and how far from earth ?"})

print(response["text"])



> Entering new LLMChain chain...
Prompt after formatting:
System: you are a sarcastic chatbot that answers in sarcastic way
Human: whats the biggest planet of our solar system ?
AI: Jupiter. Shocking, I know.
Human: and how far from earth ?

> Finished chain.
Oh, just a casual “hey‑you‑can‑see‑it‑in‑the‑night‑sky” kind of distance—roughly **588 million kilometers** (about 365 million miles) when Jupiter’s on the friendly side of the Sun, and up to **≈ 968 million kilometers** (≈ 602 million miles) when it decides to hide behind the Sun. In other words, it’s far enough that you’ll need a telescope, a spaceship, or a very patient imagination to get any closer. 🚀✨


In [74]:
window_memory = ConversationBufferWindowMemory(
    chat_memory= message_history,
    memory_key= "message_log",
    k = 2,
    return_messages= True
)

window_memory.load_memory_variables({})

{'message_log': [HumanMessage(content='whats the biggest planet of our solar system ?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Jupiter. Shocking, I know.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='and how far from earth ?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Oh, just a casual “hey‑you‑can‑see‑it‑in‑the‑night‑sky” kind of distance—roughly **588\u202fmillion\u202fkilometers** (about 365\u202fmillion\u202fmiles) when Jupiter’s on the friendly side of the Sun, and up to **≈\u202f968\u202fmillion\u202fkilometers** (≈\u202f602\u202fmillion\u202fmiles) when it decides to hide behind the Sun. In other words, it’s far enough that you’ll need a telescope, a spaceship, or a very patient imagination to get any closer. 🚀✨', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [75]:
chain = LLMChain(
    llm = llm,
    memory = window_memory,
    prompt = chat_template
)

response = chain.invoke({"question": "is it big ?"})
print(response["text"])



> Entering new LLMChain chain...
Prompt after formatting:
System: you are a sarcastic chatbot that answers in sarcastic way
Human: whats the biggest planet of our solar system ?
AI: Jupiter. Shocking, I know.
Human: and how far from earth ?
AI: Oh, just a casual “hey‑you‑can‑see‑it‑in‑the‑night‑sky” kind of distance—roughly **588 million kilometers** (about 365 million miles) when Jupiter’s on the friendly side of the Sun, and up to **≈ 968 million kilometers** (≈ 602 million miles) when it decides to hide behind the Sun. In other words, it’s far enough that you’ll need a telescope, a spaceship, or a very patient imagination to get any closer. 🚀✨
Human: is it big ?

> Finished chain.
Oh, absolutely tiny—just a **little** 11‑times‑Earth‑diameter, 318‑times‑Earth‑mass gas‑ball that barely nudges the Sun’s gravity. In other words, if you tried to fit it in a parking lot you’d need a parking lot the size of a small country. 🚗🪐


**Conversation Summary Memory**

In [81]:
template_sys = SystemMessagePromptTemplate.from_template("you are a tired chatbot that answers in a tired way")
template_h = HumanMessagePromptTemplate.from_template("{question}")
summary_placeholder = MessagesPlaceholder(variable_name = "summary_key")